In [ ]:
!pip install polars hmmlearn polars skimpy xgboost catboost lightgbm duckdb  arch statsmodels skimpy mlflow

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
import joblib
import mlflow.sklearn
import mlflow
import plotly.express as px

from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
from arch import arch_model
from statsmodels.tsa.holtwinters import Holt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import *
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from hmmlearn.hmm import GaussianHMM
from sklearn.pipeline import Pipeline
from sklearn.metrics import *
from sklearn.preprocessing import *
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.model_selection import BaseCrossValidator
from sklearn.base import clone
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
train = (
    pl.scan_parquet("/content/drive/MyDrive/prc-2026-datasets/taxi_departures.parquet")
    .collect()
)

print(train.shape)
print(train.schema)

(2085047, 14)
Schema({'MVT_ID_mvt': Float64, 'FLIGHT_ID_mvt': Float64, 'FLIGHT_mvt': String, 'FLIGHT_RULE_mvt': String, 'ADEP_mvt': String, 'ADES_mvt': String, 'PHASE_mvt': String, 'MVT_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'BLOCK_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'SCHED_TIME_UTC_mvt': Datetime(time_unit='us', time_zone='UTC'), 'AIRCRAFT_TYPE_mvt': String, 'RUNWAY_mvt': String, 'STAND_mvt': String, 'TAXITIME_SEC_mvt': Int32})


In [ ]:
test = pl.read_csv("/content/drive/MyDrive/prc-2026-datasets/merged_prc_data_test.csv")
test_ids = test["MVT_ID_mvt"]

In [ ]:
#Checking the Differences
set(train.columns) - set(test.columns)

{'TAXITIME_SEC_mvt'}

In [ ]:
def engineer_markov_features(df: pl.DataFrame) -> pl.DataFrame:

    dtype = df.schema["MVT_TIME_UTC_mvt"]

    if dtype == pl.String:
        df = df.with_columns(
            pl.col("MVT_TIME_UTC_mvt")
            .str.to_datetime(
                strict=False,
                format="%Y-%m-%d %H:%M:%S%z",
            )
            .alias("_mvt_dt")
        )
    elif isinstance(dtype, pl.Datetime):
        df = df.with_columns(
            pl.col("MVT_TIME_UTC_mvt").alias("_mvt_dt")
        )
    else:
        df = df.with_columns(
            pl.col("MVT_TIME_UTC_mvt")
            .cast(pl.Datetime("us", "UTC"), strict=False)
            .alias("_mvt_dt")
        )

    df = df.with_columns(
        pl.col("_mvt_dt")
        .dt.convert_time_zone("Europe/Berlin")
        .alias("local_datetime")
    )

    df = df.with_columns([
        pl.col("local_datetime").dt.hour().alias("local_hour"),
        pl.col("local_datetime").dt.minute().alias("local_minute"),
        pl.col("local_datetime").dt.second().alias("local_second"),
        pl.col("local_datetime").dt.weekday().alias("local_weekday"),
        pl.col("local_datetime").dt.month().alias("local_month"),
        pl.col("local_datetime").dt.day().alias("local_day"),
        pl.col("local_datetime").dt.week().alias("local_week"),
        pl.col("local_datetime").dt.ordinal_day().alias("local_day_of_year"),
    ])

    df = df.with_columns([
        (
            pl.col("local_hour") * 60
            + pl.col("local_minute")
        ).alias("time_of_day_min"),

        (
            pl.col("local_hour") * 3600
            + pl.col("local_minute") * 60
            + pl.col("local_second")
        ).alias("seconds_since_midnight"),

        (pl.col("local_weekday") >= 6)
        .cast(pl.Int8)
        .alias("is_weekend"),
    ])

    pi = np.pi

    df = df.with_columns([
        (
            2 * pi * pl.col("seconds_since_midnight") / 86400
        ).sin().alias("hour_sin"),

        (
            2 * pi * pl.col("seconds_since_midnight") / 86400
        ).cos().alias("hour_cos"),

        (
            2 * pi * (pl.col("local_weekday") - 1) / 7
        ).sin().alias("weekday_sin"),

        (
            2 * pi * (pl.col("local_weekday") - 1) / 7
        ).cos().alias("weekday_cos"),

        (
            2 * pi * (pl.col("local_month") - 1) / 12
        ).sin().alias("month_sin"),

        (
            2 * pi * (pl.col("local_month") - 1) / 12
        ).cos().alias("month_cos"),
    ])

    df = df.with_columns([
        pl.col("SCHED_TIME_UTC_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_schedule"),

        pl.col("RUNWAY_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_runway"),

        pl.col("STAND_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_stand"),

        pl.col("BLOCK_TIME_UTC_mvt")
        .is_not_null()
        .cast(pl.Int8)
        .alias("has_block_time"),
    ])

    df = df.with_columns([
        (
            pl.col("ADEP_mvt").cast(pl.String)
            + "→"
            + pl.col("ADES_mvt").cast(pl.String)
        ).alias("route"),

        (
            pl.col("ADEP_mvt").cast(pl.String)
            + "_"
            + pl.col("local_hour").cast(pl.String)
        ).alias("airport_hour"),

        (
            pl.col("ADEP_mvt").cast(pl.String)
            + "_"
            + pl.col("local_weekday").cast(pl.String)
        ).alias("airport_weekday"),

        (
            pl.col("AIRCRAFT_TYPE_mvt").cast(pl.String)
            + "_"
            + pl.col("ADEP_mvt").cast(pl.String)
        ).alias("aircraft_airport"),

        (
            pl.col("RUNWAY_mvt").cast(pl.String)
            + "_"
            + pl.col("ADEP_mvt").cast(pl.String)
        ).alias("runway_airport"),

        (
            pl.col("STAND_mvt").cast(pl.String)
            + "_"
            + pl.col("ADEP_mvt").cast(pl.String)
        ).alias("stand_airport"),
    ])

    df = df.with_columns([
        pl.col("local_hour")
        .is_between(6, 9)
        .cast(pl.Int8)
        .alias("is_morning_peak"),

        pl.col("local_hour")
        .is_between(16, 19)
        .cast(pl.Int8)
        .alias("is_evening_peak"),

        (
            pl.col("local_hour").is_between(22, 23)
            | pl.col("local_hour").is_between(0, 5)
        )
        .cast(pl.Int8)
        .alias("is_overnight"),

        pl.col("local_hour")
        .is_between(10, 15)
        .cast(pl.Int8)
        .alias("is_midday"),
    ])

    df = df.with_columns([
        (pl.col("time_of_day_min") / 1440)
        .alias("time_fraction_day"),

        (pl.col("local_day_of_year") / 365)
        .alias("time_fraction_year"),

        (pl.col("local_week") / 52)
        .alias("time_fraction_week"),
    ])

    # Preserve original row order for later alignment.
    df = df.with_row_index("_original_position")

    # Validate sequence metadata before sorting.
    if df["ADEP_mvt"].null_count() > 0:
        raise ValueError("ADEP_mvt contains null values.")

    if df["local_datetime"].null_count() > 0:
        raise ValueError("local_datetime contains null values.")

    # Chronological order within each departure airport.
    df = df.sort(
        ["ADEP_mvt", "local_datetime", "_original_position"],
        maintain_order=True,
    )

    return df.drop("_mvt_dt")


def get_airport_sequence_order(
    df: pl.DataFrame,
    airport_col: str = "ADEP_mvt",
):
    """
    Get sorted row positions and sequence lengths.
    Input must already be sorted by airport and local_datetime.
    """
    if "_original_position" not in df.columns:
        raise ValueError("Missing _original_position column.")

    order = df["_original_position"].to_numpy()

    lengths = (
        df.group_by(airport_col, maintain_order=True)
        .len()["len"]
        .to_numpy()
    )

    return order, lengths


def fit_airport_hmm(
    hmm,
    H: np.ndarray,
    order: np.ndarray,
    lengths: np.ndarray,
):
    """
    Fit one shared HMM across independent airport sequences.
    """
    H = np.asarray(H, dtype=np.float64)

    if H.ndim != 2:
        raise ValueError("H must be a two-dimensional array.")

    if len(H) != len(order):
        raise ValueError("H and order must have the same number of rows.")

    if not np.isfinite(H).all():
        raise ValueError("H contains NaN or infinite values.")

    H_sorted = H[order]
    hmm.fit(H_sorted, lengths=lengths)

    return hmm


def predict_airport_posteriors(
    hmm,
    H: np.ndarray,
    order: np.ndarray,
    lengths: np.ndarray,
):
    """
    Infer posterior state probabilities separately per airport,
    then restore the original row order.
    """
    H = np.asarray(H, dtype=np.float64)

    if H.ndim != 2:
        raise ValueError("H must be a two-dimensional array.")

    if len(H) != len(order):
        raise ValueError("H and order must have the same number of rows.")

    if not np.isfinite(H).all():
        raise ValueError("H contains NaN or infinite values.")

    H_sorted = H[order]

    gamma_sorted = np.empty(
        (len(H), hmm.n_components),
        dtype=np.float64,
    )

    start = 0

    for length in lengths:
        end = start + int(length)

        gamma_sorted[start:end] = hmm.predict_proba(
            H_sorted[start:end]
        )

        start = end

    if start != len(H):
        raise ValueError("Sequence lengths do not sum to number of rows.")

    gamma = np.empty_like(gamma_sorted)
    gamma[order] = gamma_sorted

    return gamma

In [ ]:
train = engineer_markov_features(train)
test = engineer_markov_features(test)

train_order, train_lengths = get_airport_sequence_order(train)
test_order, test_lengths = get_airport_sequence_order(test)

print(train.shape)
print(test.shape)

print("Train airport sequences:", len(train_lengths))
print("Test airport sequences:", len(test_lengths))

(2085047, 50)
(344841, 65)
Train airport sequences: 10
Test airport sequences: 10


In [ ]:
#Checking the Differences
set(test.columns) - set(train.columns)

{'ADEP_flt',
 'ADES_FILED_flt',
 'ADES_flt',
 'AIRCRAFT_OPERATOR_flt',
 'AIRCRAFT_TYPE_flt',
 'AOBT_3_flt',
 'ARVT_1_flt',
 'ARVT_3_flt',
 'CALLSIGN_flt',
 'EOBT_1_flt',
 'FLIGHT_RULE_flt',
 'FLIGHT_TYPE_flt',
 'IOBT_flt',
 'LOBT_flt',
 'MARKET_SEGMENT_flt',
 'WK_TBL_CAT_flt'}

In [ ]:
cols = ['ADEP_flt',
 'ADES_FILED_flt',
 'ADES_flt',
 'AIRCRAFT_OPERATOR_flt',
 'AIRCRAFT_TYPE_flt',
 'AOBT_3_flt',
 'ARVT_1_flt',
 'ARVT_3_flt',
 'CALLSIGN_flt',
 'EOBT_1_flt',
 'FLIGHT_RULE_flt',
 'FLIGHT_TYPE_flt',
 'IOBT_flt',
 'LOBT_flt',
 'MARKET_SEGMENT_flt',
 'WK_TBL_CAT_flt']
test = test.drop(cols)

In [ ]:
#Checking the Differences
set(train.columns) - set(test.columns)

{'TAXITIME_SEC_mvt'}

In [ ]:
print(train.shape)
print(test.shape)

(2085047, 50)
(344841, 49)


In [ ]:
categorical_cols = [
    "FLIGHT_RULE_mvt",
    "ADEP_mvt",
    "ADES_mvt",
    "PHASE_mvt",
    "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt",
    "STAND_mvt",
    "route",
    "airport_hour",
    "airport_weekday",
    "aircraft_airport",
    "runway_airport",
    "stand_airport",
]

encoders = {}

for col in categorical_cols:

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )

    train_values = (
        train
        .select(col)
        .to_series()
        .cast(pl.String)
        .fill_null("__MISSING__")
        .to_numpy()
        .reshape(-1, 1)
    )

    test_values = (
        test
        .select(col)
        .to_series()
        .cast(pl.String)
        .fill_null("__MISSING__")
        .to_numpy()
        .reshape(-1, 1)
    )

    encoder.fit(train_values)

    train = train.with_columns(
        pl.Series(
            col,
            encoder.transform(train_values).ravel().astype(np.float32),
        )
    )

    test = test.with_columns(
        pl.Series(
            col,
            encoder.transform(test_values).ravel().astype(np.float32),
        )
    )

    encoders[col] = encoder

In [ ]:
for col in categorical_cols:

    encoder = encoders[col]

    train_values = (
        train
        .select(col)
        .to_series()
        .cast(pl.String)
        .fill_null("__MISSING__")
        .to_numpy()
        .reshape(-1, 1)
    )

    test_values = (
        test
        .select(col)
        .to_series()
        .cast(pl.String)
        .fill_null("__MISSING__")
        .to_numpy()
        .reshape(-1, 1)
    )

    train_encoded = encoder.transform(train_values).ravel()
    test_encoded = encoder.transform(test_values).ravel()

    train = train.with_columns(
        pl.Series(col, train_encoded.astype(np.int32))
    )

    test = test.with_columns(
        pl.Series(col, test_encoded.astype(np.int32))
    )

In [ ]:
for df_name, df in [
    ("train", train),
    ("test", test),
]:

    numeric_cols = [
        col
        for col, dtype in df.schema.items()
        if dtype.is_numeric()
    ]

    df = df.with_columns([
        pl.col(col).fill_null(0)
        for col in numeric_cols
    ])

    if df_name == "train":
        train = df
    else:
        test = df

In [ ]:
train

_original_position,MVT_ID_mvt,FLIGHT_ID_mvt,FLIGHT_mvt,FLIGHT_RULE_mvt,ADEP_mvt,ADES_mvt,PHASE_mvt,MVT_TIME_UTC_mvt,BLOCK_TIME_UTC_mvt,SCHED_TIME_UTC_mvt,AIRCRAFT_TYPE_mvt,RUNWAY_mvt,STAND_mvt,TAXITIME_SEC_mvt,local_datetime,local_hour,local_minute,local_second,local_weekday,local_month,local_day,local_week,local_day_of_year,time_of_day_min,seconds_since_midnight,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,has_schedule,has_runway,has_stand,has_block_time,route,airport_hour,airport_weekday,aircraft_airport,runway_airport,stand_airport,is_morning_peak,is_evening_peak,is_overnight,is_midday,time_fraction_day,time_fraction_year,time_fraction_week
u32,f64,f64,str,i32,i32,i32,i32,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",i32,i32,i32,i32,"datetime[μs, Europe/Berlin]",i8,i8,i8,i8,i8,i8,i8,i16,i8,i16,i8,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i32,i32,i32,i32,i32,i32,i8,i8,i8,i8,f64,f64,f64
4277,1.82230152e8,2.80367414e8,"""TP575""",-1,-1,-1,-1,2025-01-01 05:26:46 UTC,2025-01-01 05:13:50 UTC,2025-01-01 05:00:00 UTC,-1,-1,-1,776,2025-01-01 06:26:46 CET,6,26,46,3,1,1,1,1,-126,21670,0,0.999987,-0.005091,0.974928,-0.222521,0.0,1.0,1,1,1,1,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.0875,0.00274,0.019231
4278,1.82230153e8,2.80367409e8,"""TUI2138""",-1,-1,-1,-1,2025-01-01 05:28:12 UTC,2025-01-01 05:12:17 UTC,2025-01-01 05:00:00 UTC,-1,-1,-1,955,2025-01-01 06:28:12 CET,6,28,12,3,1,1,1,1,-124,21500,0,0.999974,0.007272,0.974928,-0.222521,0.0,1.0,1,1,1,1,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.086111,0.00274,0.019231
4287,1.82230154e8,2.80367885e8,"""LH002""",-1,-1,-1,-1,2025-01-01 05:41:22 UTC,2025-01-01 05:30:03 UTC,2025-01-01 05:30:00 UTC,-1,-1,-1,679,2025-01-01 06:41:22 CET,6,41,22,3,1,1,1,1,-111,21522,0,0.999984,0.005672,0.974928,-0.222521,0.0,1.0,1,1,1,1,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.077083,0.00274,0.019231
4288,1.82230156e8,2.80368234e8,"""KL1814""",-1,-1,-1,-1,2025-01-01 05:57:45 UTC,2025-01-01 05:50:55 UTC,2025-01-01 05:55:00 UTC,-1,-1,-1,410,2025-01-01 06:57:45 CET,6,57,45,3,1,1,1,1,-95,21737,0,0.99995,-0.009963,0.974928,-0.222521,0.0,1.0,1,1,1,1,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.065972,0.00274,0.019231
4289,1.82230159e8,2.80368342e8,"""LH8402""",-1,-1,-1,-1,2025-01-01 06:03:00 UTC,2025-01-01 05:48:35 UTC,2025-01-01 06:00:00 UTC,-1,-1,-1,865,2025-01-01 07:03:00 CET,7,3,0,3,1,1,1,1,-89,25124,0,0.967342,-0.253477,0.974928,-0.222521,0.0,1.0,1,1,1,1,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.061806,0.00274,0.019231
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2084839,2.01598972e8,2.94565604e8,"""SVA260""",-1,-1,-1,-1,2025-12-31 20:08:04 UTC,2025-12-31 19:52:55 UTC,2025-12-31 19:25:00 UTC,-1,-1,-1,909,2025-12-31 21:08:04 CET,21,8,4,3,12,31,1,365,-12,10036,0,0.666749,0.745282,0.974928,-0.222521,-0.5,0.866025,1,1,1,1,-1,-1,-1,-1,-1,-1,0,0,0,0,-0.008333,1.0,0.019231
2084840,2.01598974e8,2.94566079e8,"""CFG2368""",-1,-1,-1,-1,2025-12-31 20:22:02 UTC,2025-12-31 20:00:58 UTC,2025-12-31 19:15:00 UTC,-1,-1,-1,1264,2025-12-31 21:22:02 CET,21,22,2,3,12,31,1,365,2,10106,0,0.670535,0.741878,0.974928,-0.222521,-0.5,0.866025,1,1,1,1,-1,-1,-1,-1,-1,-1,0,0,0,0,0.001389,1.0,0.019231
2084889,2.01598975e8,2.94566262e8,"""THY6047""",-1,-1,-1,-1,2025-12-31 20:49:01 UTC,2025-12-31 20:34:59 UTC,2025-12-31 19:10:00 UTC,-1,-1,-1,842,2025-12-31 21:49:01 CET,21,49,1,3,12,31,1,365,29,10189,0,0.675,0.737818,0.974928,-0.222521,-0.5,0.866025,1,1,1,1,-1,-1,-1,-1,-1,-1,0,0,0,0,0.020139,1.0,0.019231


In [ ]:
test

_original_position,MVT_ID_mvt,FLIGHT_ID_mvt,FLIGHT_mvt,FLIGHT_RULE_mvt,ADEP_mvt,ADES_mvt,PHASE_mvt,MVT_TIME_UTC_mvt,BLOCK_TIME_UTC_mvt,SCHED_TIME_UTC_mvt,AIRCRAFT_TYPE_mvt,RUNWAY_mvt,STAND_mvt,local_datetime,local_hour,local_minute,local_second,local_weekday,local_month,local_day,local_week,local_day_of_year,time_of_day_min,seconds_since_midnight,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos,has_schedule,has_runway,has_stand,has_block_time,route,airport_hour,airport_weekday,aircraft_airport,runway_airport,stand_airport,is_morning_peak,is_evening_peak,is_overnight,is_midday,time_fraction_day,time_fraction_year,time_fraction_week
u32,f64,f64,str,i32,i32,i32,i32,str,str,str,i32,i32,i32,"datetime[μs, Europe/Berlin]",i8,i8,i8,i8,i8,i8,i8,i16,i8,i16,i8,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i32,i32,i32,i32,i32,i32,i8,i8,i8,i8,f64,f64,f64
590,2.02499285e8,2.94582427e8,"""TP575""",-1,-1,-1,-1,"""2026-01-01 05:06:14+00:00""",null,"""2026-01-01 05:00:00+00:00""",-1,-1,-1,2026-01-01 06:06:14 CET,6,6,14,4,1,1,1,1,110,21718,0,0.999963,-0.008581,0.433884,-0.900969,0.0,1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,1,0,0,0,0.076389,0.00274,0.019231
591,2.02499282e8,2.94582458e8,"""DE2290""",-1,-1,-1,-1,"""2026-01-01 05:08:50+00:00""",null,"""2025-12-31 20:55:00+00:00""",-1,-1,-1,2026-01-01 06:08:50 CET,6,8,50,4,1,1,1,1,112,21618,0,0.999999,-0.001309,0.433884,-0.900969,0.0,1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,1,0,0,0,0.077778,0.00274,0.019231
592,2.02499284e8,2.94582323e8,"""TUI2138""",-1,-1,-1,-1,"""2026-01-01 05:23:55+00:00""",null,"""2026-01-01 05:00:00+00:00""",-1,-1,-1,2026-01-01 06:23:55 CET,6,23,55,4,1,1,1,1,127,21755,0,0.999936,-0.011272,0.433884,-0.900969,0.0,1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,1,0,0,0,0.088194,0.00274,0.019231
593,2.02499286e8,2.94582825e8,"""LH002""",-1,-1,-1,-1,"""2026-01-01 05:38:48+00:00""",null,"""2026-01-01 05:30:00+00:00""",-1,-1,-1,2026-01-01 06:38:48 CET,6,38,48,4,1,1,1,1,-114,21624,0,0.999998,-0.001745,0.433884,-0.900969,0.0,1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.079167,0.00274,0.019231
594,2.02499287e8,2.94582876e8,"""IB1332""",-1,-1,-1,-1,"""2026-01-01 05:44:44+00:00""",null,"""2026-01-01 05:30:00+00:00""",-1,-1,-1,2026-01-01 06:44:44 CET,6,44,44,4,1,1,1,1,-108,21724,0,0.999959,-0.009017,0.433884,-0.900969,0.0,1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,1,0,0,0,-0.075,0.00274,0.019231
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
305550,2.13442781e8,3.02954969e8,"""JAV272""",-1,-1,-1,-1,"""2026-07-31 20:43:02+00:00""",null,"""2026-07-31 19:45:00+00:00""",-1,-1,-1,2026-07-31 22:43:02 CEST,22,43,2,5,7,31,31,212,83,13686,0,0.838908,0.544273,-0.433884,-0.900969,1.2246e-16,-1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,0,0,1,0,0.057639,0.580822,0.596154
305551,2.13442782e8,3.02955876e8,"""VLG3181""",-1,-1,-1,-1,"""2026-07-31 20:45:00+00:00""",null,"""2026-07-31 20:20:00+00:00""",-1,-1,-1,2026-07-31 22:45:00 CEST,22,45,0,5,7,31,31,212,85,13548,0,0.833404,0.552664,-0.433884,-0.900969,1.2246e-16,-1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,0,0,1,0,0.059028,0.580822,0.596154
305552,2.13442783e8,3.02955862e8,"""PBD994""",-1,-1,-1,-1,"""2026-07-31 20:47:58+00:00""",null,"""2026-07-31 13:10:00+00:00""",-1,-1,-1,2026-07-31 22:47:58 CEST,22,47,58,5,7,31,31,212,87,13726,0,0.840488,0.54183,-0.433884,-0.900969,1.2246e-16,-1.0,1,1,1,0,-1,-1,-1,-1,-1,-1,0,0,1,0,0.060417,0.580822,0.596154


In [ ]:
target = "TAXITIME_SEC_mvt"

drop_cols = [
    "MVT_ID_mvt",
    "FLIGHT_ID_mvt",
    "FLIGHT_mvt",
    "MVT_TIME_UTC_mvt",
    "BLOCK_TIME_UTC_mvt",
    "SCHED_TIME_UTC_mvt",
    target,
]

X = train.drop(drop_cols)
y = train[target]

# Split chronologically within each departure airport.
split_fraction = 0.8

train_positions = []
test_positions = []

for airport in train["ADEP_mvt"].unique(maintain_order=True).to_list():
    airport_positions = (
        train.with_row_index("_position")
        .filter(pl.col("ADEP_mvt") == airport)
        .sort(["local_datetime", "_position"])
        ["_position"]
        .to_numpy()
    )

    split_idx = int(len(airport_positions) * split_fraction)

    train_positions.extend(airport_positions[:split_idx])
    test_positions.extend(airport_positions[split_idx:])

train_positions = np.asarray(train_positions, dtype=np.int64)
test_positions = np.asarray(test_positions, dtype=np.int64)

X_train = X[train_positions]
X_test = X[test_positions]

y_train = y[train_positions]
y_test = y[test_positions]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1668037, 43)
X_test: (417010, 43)
y_train: (1668037,)
y_test: (417010,)


In [ ]:
class CPCV(BaseCrossValidator):

    def __init__(
        self,
        n_groups=10,
        n_test_groups=2,
        purge_minutes=60,
        embargo_minutes=30,
        time_col="local_datetime",
    ):
        self.n_groups = n_groups
        self.n_test_groups = n_test_groups
        self.purge_minutes = purge_minutes
        self.embargo_minutes = embargo_minutes
        self.time_col = time_col

    def get_n_splits(self, X=None, y=None, groups=None):
        from math import comb
        return comb(self.n_groups, self.n_test_groups)

    def split(self, X, y=None, groups=None):

        if not isinstance(X, pl.DataFrame):
            raise TypeError("CPCV expects a Polars DataFrame")

        if self.time_col not in X.columns:
            raise ValueError(
                f"{self.time_col!r} not found in DataFrame"
            )

        n = X.height

        if n < self.n_groups:
            raise ValueError(
                f"n_samples={n} is smaller than "
                f"n_groups={self.n_groups}"
            )

        # Original positional indices
        positions = np.arange(n)

        # Datetime values
        times = X.get_column(self.time_col).to_numpy()

        # Chronological ordering
        order = np.argsort(times)

        ordered_positions = positions[order]
        ordered_times = times[order]

        # Equal-sized chronological groups
        groups_idx = np.array_split(
            ordered_positions,
            self.n_groups,
        )

        # All combinations of test groups
        from itertools import combinations

        for test_groups in combinations(
            range(self.n_groups),
            self.n_test_groups,
        ):

            test_groups = tuple(test_groups)

            test_idx = np.concatenate(
                [groups_idx[g] for g in test_groups]
            )

            test_idx = np.sort(test_idx)

            test_times = times[test_idx]

            test_start = test_times.min()
            test_end = test_times.max()

            purge_delta = np.timedelta64(
                self.purge_minutes,
                "m",
            )

            embargo_delta = np.timedelta64(
                self.embargo_minutes,
                "m",
            )

            purge_start = test_start - purge_delta
            embargo_end = test_end + embargo_delta

            # Start with every observation not in test
            is_test = np.zeros(
                n,
                dtype=bool,
            )

            is_test[test_idx] = True

            # Purge observations immediately before test
            purge_mask = (
                (times >= purge_start)
                & (times < test_start)
            )

            # Embargo observations immediately after test
            embargo_mask = (
                (times > test_end)
                & (times <= embargo_end)
            )

            excluded = (
                is_test
                | purge_mask
                | embargo_mask
            )

            train_idx = positions[~excluded]

            train_idx = np.sort(train_idx)

            # Safety checks
            if len(train_idx) == 0:
                raise RuntimeError(
                    f"CPCV generated an empty training fold. "
                    f"Test groups={test_groups}, "
                    f"test_size={len(test_idx)}"
                )

            if len(test_idx) == 0:
                raise RuntimeError(
                    f"CPCV generated an empty test fold. "
                    f"Test groups={test_groups}"
                )

            yield train_idx, test_idx

In [ ]:

class MarkovSwitchingRegressor(BaseEstimator, RegressorMixin):

    def __init__(
        self,
        base_estimator,
        n_regimes=3,
        log_target=False,
        time_col="local_datetime",
        group_col="ADEP_mvt",
        hmm_features=None,
        hmm_n_iter=100,
        hmm_tol=1e-3,
        covariance_type="diag",
        random_state=42,
    ):
        self.base_estimator = base_estimator
        self.n_regimes = n_regimes
        self.log_target = log_target
        self.time_col = time_col
        self.group_col = group_col
        self.hmm_features = hmm_features
        self.hmm_n_iter = hmm_n_iter
        self.hmm_tol = hmm_tol
        self.covariance_type = covariance_type
        self.random_state = random_state

    def _check_X(self, X):
        if not isinstance(X, pl.DataFrame):
            raise TypeError("X must be a Polars DataFrame.")

        required = [self.time_col, self.group_col]
        missing = [c for c in required if c not in X.columns]

        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        if X[self.time_col].null_count() > 0:
            raise ValueError(f"{self.time_col} contains null values.")

        if X[self.group_col].null_count() > 0:
            raise ValueError(f"{self.group_col} contains null values.")

        return X

    def _sort_sequences(self, X):
        X = X.with_row_index("_original_index")

        X = X.sort(
            [self.group_col, self.time_col, "_original_index"],
            maintain_order=True,
        )

        lengths = (
            X.group_by(self.group_col, maintain_order=True)
            .len()
            .get_column("len")
            .to_numpy()
            .astype(int)
        )

        if lengths.sum() != X.height:
            raise ValueError("Sequence lengths do not match row count.")

        return X, lengths

    def _get_numeric_columns(self, X):
        excluded = {
            self.time_col,
            self.group_col,
            "_original_index",
            "_original_position",
        }

        return [
            c for c in X.columns
            if c not in excluded and X.schema[c].is_numeric()
        ]

    def _raw_hmm_features(self, X):
        if self.hmm_features is None:
            columns = self._get_numeric_columns(X)
        else:
            missing = [
                c for c in self.hmm_features
                if c not in X.columns
            ]

            if missing:
                raise ValueError(f"Missing HMM features: {missing}")

            columns = list(self.hmm_features)

        if not columns:
            raise ValueError("No HMM features available.")

        H = (
            X.select(columns)
            .to_numpy()
            .astype(np.float64)
        )

        H = np.nan_to_num(
            H,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return H, columns

    def _scale_hmm_features(self, H, fit=False):
        if fit:
            self.hmm_mean_ = H.mean(axis=0)
            self.hmm_std_ = H.std(axis=0)
            self.hmm_std_[self.hmm_std_ < 1e-8] = 1.0

        H = (H - self.hmm_mean_) / self.hmm_std_

        return np.clip(H, -8.0, 8.0)

    def _prepare_regression_features(self, X, fit=False):
        if fit:
            columns = self._get_numeric_columns(X)

            if not columns:
                raise ValueError(
                    "No numeric regression features available."
                )

            self.regression_feature_names_ = columns
        else:
            columns = self.regression_feature_names_

            missing = [c for c in columns if c not in X.columns]
            if missing:
                raise ValueError(
                    f"Missing regression features: {missing}"
                )

        Z = (
            X.select(columns)
            .to_numpy()
            .astype(np.float64)
        )

        Z = np.nan_to_num(
            Z,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

        return Z

    def _predict_proba_by_sequence(self, H, lengths):
        probabilities = np.empty(
            (len(H), self.n_regimes),
            dtype=np.float64,
        )

        start = 0

        for length in lengths:
            end = start + int(length)

            if length <= 0:
                raise ValueError("Encountered an empty sequence.")

            probabilities[start:end] = (
                self.hmm_.predict_proba(H[start:end])
            )

            start = end

        if start != len(H):
            raise ValueError(
                "Sequence lengths do not sum to number of rows."
            )

        return probabilities

    def fit(self, X, y):
        X = self._check_X(X)

        y = np.asarray(y, dtype=np.float64).reshape(-1)

        if len(X) != len(y):
            raise ValueError("X and y must have identical lengths.")

        if not np.isfinite(y).all():
            raise ValueError("y contains NaN or infinite values.")

        if self.log_target and np.any(y < 0):
            raise ValueError(
                "log_target=True requires non-negative target values."
            )

        X_sorted, lengths = self._sort_sequences(X)

        original_index = (
            X_sorted.get_column("_original_index").to_numpy()
        )
        y_sorted = y[original_index]

        H, self.hmm_feature_names_ = self._raw_hmm_features(
            X_sorted
        )
        H = self._scale_hmm_features(H, fit=True)

        self.hmm_ = GaussianHMM(
            n_components=self.n_regimes,
            covariance_type=self.covariance_type,
            n_iter=self.hmm_n_iter,
            tol=self.hmm_tol,
            random_state=self.random_state,
        )

        self.hmm_.fit(H, lengths=lengths)

        responsibilities = self._predict_proba_by_sequence(
            H, lengths
        )

        Z = self._prepare_regression_features(
            X_sorted, fit=True
        )

        if self.log_target:
            y_model = np.log1p(y_sorted)
        else:
            y_model = y_sorted

        self.regressors_ = []
        self.regime_weight_sums_ = []

        for regime in range(self.n_regimes):
            model = clone(self.base_estimator)
            weights = responsibilities[:, regime]

            weight_sum = weights.sum()
            self.regime_weight_sums_.append(weight_sum)

            if not np.isfinite(weight_sum) or weight_sum <= 1e-8:
                raise ValueError(
                    f"Regime {regime} has insufficient posterior weight."
                )

            model.fit(
                Z,
                y_model,
                sample_weight=weights,
            )

            self.regressors_.append(model)

        self.transition_matrix_ = self.hmm_.transmat_.copy()
        self.regime_occupancy_ = responsibilities.mean(axis=0)
        self.is_fitted_ = True

        return self

    def _get_regime_probabilities(self, X_sorted, lengths):
        H, _ = self._raw_hmm_features(X_sorted)

        H = self._scale_hmm_features(H, fit=False)

        return self._predict_proba_by_sequence(H, lengths)

    def predict_regimes(self, X):
        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Model has not been fitted.")

        X = self._check_X(X)
        X_sorted, lengths = self._sort_sequences(X)

        probabilities = self._get_regime_probabilities(
            X_sorted, lengths
        )

        original_index = (
            X_sorted.get_column("_original_index").to_numpy()
        )

        output = np.empty_like(probabilities)
        output[original_index] = probabilities

        return output

    def predict(self, X):
        if not getattr(self, "is_fitted_", False):
            raise RuntimeError("Model has not been fitted.")

        X = self._check_X(X)
        X_sorted, lengths = self._sort_sequences(X)

        probabilities = self._get_regime_probabilities(
            X_sorted, lengths
        )

        Z = self._prepare_regression_features(
            X_sorted, fit=False
        )

        predictions = np.column_stack([
            model.predict(Z)
            for model in self.regressors_
        ])

        prediction = np.sum(
            probabilities * predictions,
            axis=1,
        )

        if self.log_target:
            prediction = np.expm1(
                np.clip(prediction, -20, 20)
            )

        prediction = np.maximum(prediction, 0.0)

        original_index = (
            X_sorted.get_column("_original_index").to_numpy()
        )

        output = np.empty(len(prediction), dtype=np.float64)
        output[original_index] = prediction

        return output

In [ ]:
## Models
xgb = XGBRegressor(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

lgbm = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="regression",
    n_jobs=-1,
    random_state=42,
)

cat = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=False,
    random_seed=42,
)

In [ ]:
hmm_xgb = Pipeline([
    ("model", MarkovSwitchingRegressor(
      base_estimator=xgb,
      n_regimes=3,
      log_target=False,
      random_state=42,
    ))
])

hmm_lgbm = Pipeline([
    ("model", MarkovSwitchingRegressor(
      base_estimator=lgbm,
      n_regimes=3,
      log_target=False,
      random_state=42,
    ))
])

hmm_cat = Pipeline([
    ("model", MarkovSwitchingRegressor(
      base_estimator=cat,
      n_regimes=3,
      log_target=False,
      random_state=42,
    ))
])

In [ ]:
models = {
    "xgb": hmm_xgb,
    "lgbm": hmm_lgbm,
    "cat": hmm_cat
}

mlflow.set_experiment("HMM Learn Experiment")

cv = CPCV(
    n_groups=5,
    n_test_groups=2,
    purge_minutes=60,
    embargo_minutes=30,
    time_col="local_datetime",
)

results = []
all_residuals = []

for name, model in models.items():

    with mlflow.start_run(run_name=f"{name}_HMM_Experiment"):

        fold_metrics = []
        model_residuals = []

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(train)
        ):

            fold_train = train[train_idx]
            fold_test = train[test_idx]

            y_train = fold_train[target]
            y_test = fold_test[target]

            X_train = fold_train.drop(
                [
                    c for c in drop_cols
                    if c in fold_train.columns
                ]
            )

            X_test = fold_test.drop(
                [
                    c for c in drop_cols
                    if c in fold_test.columns
                ]
            )

            if "local_datetime" not in X_train.columns:
                raise ValueError(
                    "local_datetime must be retained for "
                    "Markov sequence construction."
                )

            if "ADEP_mvt" not in X_train.columns:
                raise ValueError(
                    "ADEP_mvt must be retained for "
                    "Markov sequence construction."
                )

            estimator = clone(model)

            estimator.fit(
                X_train,
                y_train.to_numpy(),
            )

            pred = estimator.predict(
                X_test
            )

            y_test_np = y_test.to_numpy()

            fold_residuals = (
                y_test_np - pred
            )

            mae = mean_absolute_error(
                y_test_np,
                pred,
            )

            rmse = np.sqrt(
                mean_squared_error(
                    y_test_np,
                    pred,
                )
            )

            r2 = r2_score(
                y_test_np,
                pred,
            )

            fold_metrics.append({
                "fold": fold,
                "mae": mae,
                "rmse": rmse,
                "r2": r2,
            })

            model_residuals.extend(
                fold_residuals.tolist()
            )

            print(
                f"{name} | "
                f"Fold {fold:02d} | "
                f"MAE={mae:.2f} | "
                f"RMSE={rmse:.2f} | "
                f"R²={r2:.4f}"
            )

        metrics_df = pl.DataFrame(
            fold_metrics
        )

        mean_mae = metrics_df["mae"].mean()
        std_mae = metrics_df["mae"].std()

        mean_rmse = metrics_df["rmse"].mean()
        std_rmse = metrics_df["rmse"].std()

        mean_r2 = metrics_df["r2"].mean()
        std_r2 = metrics_df["r2"].std()

        residuals_np = np.asarray(
            model_residuals,
            dtype=np.float64,
        )

        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)

        print(
            f"MAE  : {mean_mae:.2f} ± {std_mae:.2f}"
        )

        print(
            f"RMSE : {mean_rmse:.2f} ± {std_rmse:.2f}"
        )

        print(
            f"R²   : {mean_r2:.4f} ± {std_r2:.4f}"
        )

        print(
            f"Residual mean : "
            f"{residuals_np.mean():.2f}"
        )

        print(
            f"Residual std  : "
            f"{residuals_np.std():.2f}"
        )

        print(
            f"Residual median : "
            f"{np.median(residuals_np):.2f}"
        )

        print(
            f"Residual Q05 : "
            f"{np.quantile(residuals_np, 0.05):.2f}"
        )

        print(
            f"Residual Q95 : "
            f"{np.quantile(residuals_np, 0.95):.2f}"
        )

        print(
            f"Max |residual| : "
            f"{np.max(np.abs(residuals_np)):.2f}"
        )

        mlflow.log_metrics({
            "cv_mae_mean": mean_mae,
            "cv_mae_std": std_mae,
            "cv_rmse_mean": mean_rmse,
            "cv_rmse_std": std_rmse,
            "cv_r2_mean": mean_r2,
            "cv_r2_std": std_r2,
            "residual_mean": residuals_np.mean(),
            "residual_std": residuals_np.std(),
            "residual_median": np.median(
                residuals_np
            ),
            "residual_q05": np.quantile(
                residuals_np,
                0.05,
            ),
            "residual_q95": np.quantile(
                residuals_np,
                0.95,
            ),
            "max_abs_residual": np.max(
                np.abs(residuals_np)
            ),
        })

        results.append({
            "model": name,
            "mae_mean": mean_mae,
            "mae_std": std_mae,
            "rmse_mean": mean_rmse,
            "rmse_std": std_rmse,
            "r2_mean": mean_r2,
            "r2_std": std_r2,
        })

        all_residuals.append({
            "model": name,
            "residuals": residuals_np,
        })

        print()

xgb | Fold 00 | MAE=355.46 | RMSE=584.98 | R²=-0.2640


KeyboardInterrupt: 